In [ ]:
%matplotlib widget
import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)

import matplotlib.pyplot as plt
import os
import torch
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import torch
from gencase import *
from util import visualize, sampleParticles, generateInitialVariables, SamplingScheme
from sample import smoothState, addNoise, populateCGrid, populateUGrid, smoothValues
from util import plotState, plotInitialState
from simulation import runSimulation
from util import getCurrentTimestamp, copyWaveSystem
from argparse import ArgumentParser

import h5py
from dataset import *

In [ ]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
datasetProperties = DataSetProperties(
    skipInitialSteps= 0,
    skipFinalSteps= 0,
    temporalCoarseGrainingRate= 1,
    unrollLength= 1,
    historyLength= 0
)

simFolder = 'output'
files, S, T, N, Nx, Ny = getDatasetProperties(simFolder)
print(f'Dataset has S={S} samples, T={T} timesteps, N={N} particles, Nx={Nx}, Ny={Ny}')

In [ ]:
dataset = Dataset(datasetProperties, files, T, device)
batchSize = 1

datasetLoader = torch.utils.data.DataLoader(dataset, batch_size=batchSize, shuffle=True)
datasetIter = iter(datasetLoader)
batch = next(datasetIter)

In [ ]:
waveSystem, config, integrator, dt, history, trajectory, current = batchToSimulation(next(datasetIter))



In [ ]:
fig, axis, uPlot, vPlot, cPlot, dampPlot = plotState(
    waveSystem.systemState,
    waveSystem.waveState,
    config, config['kernel'],
    markerSize = 0.5,
    plotGrid = True,
    plotCD = False)
# fig.savefig(f'output/{folderName}/initial_state.png', dpi = args.figureDpi)

In [ ]:
particleState = waveSystem.systemState
kernel = config['kernel']

neighborhood, neighbors = evaluateNeighborhood(particleState, config['domain'], kernel, verletScale = config['neighborhood']['verletScale'], mode = SupportScheme.SuperSymmetric, priorNeighborhood=None)
particleState.numNeighbors = coo_to_csr(filterNeighborhoodByKind(particleState, neighbors.neighbors, which = 'noghost')).rowEntries


In [ ]:
from ml import *

basisTerms = 7
hiddenLayers = 2
hiddenUnits = 64
nodeFeatures = 16
activation = 'gelu'

additionalEdgeFeatures = 2 # v_i, v_j, h_i, h_j

# neighbors.get('noghost')[1].x_ij

encodedDistances = basisEncoderLayer(
    neighbors.get('noghost')[1].x_ij,
    basisTerms, 
    'ffourier',
    'cat'
)
GNN = SimpleGNN2({
    'hiddenLayers': hiddenLayers,
    'hiddenUnits': hiddenUnits,
    'featureCount': additionalEdgeFeatures,
    'coordinateFeatures': encodedDistances.shape[1],
    'nodeFeatures': nodeFeatures,
    'output': 2,
    'activation': activation,
    'basisTerms': basisTerms,
    'basis': 'ffourier',
    'encoderMode': 'cat',
    'gain': 1/10,
    'messagePassingLayers': 3,
    'finalVertexMLP': True,
}).to(device)


integrator = getIntegrator(IntegrationSchemeType.explicitEuler)
optimizer = torch.optim.Adam(GNN.parameters(), lr=1e-3)
learningRateScheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.75)



In [ ]:
import time

data = next(datasetIter)
eulerIntegrator = getIntegrator(IntegrationSchemeType.explicitEuler)
RK4Integrator = getIntegrator(IntegrationSchemeType.rungeKutta4)

trainIter = 4000
currentUnrollLength = 1
unrollLength = 1
# tq = tqdm(total=trainIter, desc='Training', position=0, leave=True)
# sleepTime = 0.1
# time.sleep(0.1)
# tq2 = tqdm(total=currentUnrollLength, desc='Unroll', position=1, leave=True)

In [ ]:
losses = []
uLosses = []
vLosses = []
for i in (tq := tqdm(range(trainIter))):
# for i in range(trainIter):
    if i % 250 == 0 and i > 0:
        currentUnrollLength = min(currentUnrollLength + 1, unrollLength)
        # tq2.n = currentUnrollLength
        # print(f'Current unroll length: {currentUnrollLength}')

    optimizer.zero_grad()

    try:
        data = next(datasetIter)
    except StopIteration:
        datasetIter = iter(datasetLoader)
        data = next(datasetIter)

    with torch.no_grad():
        waveSystem, config, integrator, dt, history, trajectory, current = batchToSimulation(data)


    loss = 0

    waveSystemNext = copyWaveSystem(waveSystem)

    uuLosses = []
    vuLosses = []

    for t in range(currentUnrollLength):
        # waveSystemNext, updates =  eulerIntegrator.function(
        #     waveSystem_,
        #     dt = dt,
        #     f = waveSystemFunction,
        #     verbose = False,
        #     config = config,
        # )
        waveSystemNext = copyWaveSystem(waveSystemNext)

        features = torch.cat([
            waveSystemNext.waveState.u.view(-1, 1),
            waveSystemNext.waveState.v.view(-1, 1),
        ], dim=-1)

        fi_nn = GNN(waveSystemNext.systemState, waveSystemNext.neighborhood, features)
        # fi_nn = torch.zeros_like(features)
        dudt = fi_nn[:, 0]
        dvdt = fi_nn[:, 1]

        # print(f'Max dudt: {torch.max(dudt)}, Max dvdt: {torch.max(dvdt)}')
        # print(f'Requires Gradient: dudt {dudt.requires_grad}, dvdt {dvdt.requires_grad}')

        waveSystemNext.waveState.u = waveSystemNext.waveState.u + dudt
        waveSystemNext.waveState.v = waveSystemNext.waveState.v + dvdt

        uLoss = torch.mean((waveSystemNext.waveState.u - trajectory[t,:,0])**2)
        vLoss = torch.mean((waveSystemNext.waveState.v - trajectory[t,:,1])**2)

        uuLosses.append(uLoss)
        vuLosses.append(vLoss)

        # print(f'Iter {i}, uLoss: {uLoss} [type: {type(uLoss)}], vLoss: {vLoss}')
        # print(uLoss)
        # print(vLoss)

    uuLosses = torch.stack(uuLosses)
    vuLosses = torch.stack(vuLosses)

    uLossTotal = sum(uuLosses) / len(uuLosses)
    vLossTotal = sum(vuLosses) / len(vuLosses)

    # print(uLossTotal, vLossTotal)

    totalLoss = uLossTotal + vLossTotal
    # print(f'Total Loss:' , totalLoss)
    totalLoss.backward()
    optimizer.step()
    learningRateScheduler.step()

    losses.append(totalLoss.detach().item())
    uLosses.append(uLossTotal.detach().item())
    vLosses.append(vLossTotal.detach().item())
    
    tq.set_description(f'Training (Loss: {totalLoss.item():.6f}, uLoss: {uLossTotal.item():.6f}, vLoss: {vLossTotal.item():.6f}) [{data[-2].cpu().item()} | {data[-1].cpu().item()}]')
    
    # tq.set_postfix({'loss': totalLoss.detach().item(), 'uLoss': uLossTotal.detach().item(), 'vLoss': vLossTotal.detach().item(), 'lr': learningRateScheduler.get_last_lr()[0]})
    # tq.update()


In [ ]:
fig, axis = plt.subplots(2,4, figsize=(16, 7), squeeze=False)

markerSize = 0.5

uInitial = visualizeParticles(fig, axis[0,0], waveSystem.systemState, config['domain'], waveSystem.waveState.u, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'managua', markerSize = markerSize, gridVisualization = False, title = 'u [Initial]')
vInitial = visualizeParticles(fig, axis[1,0], waveSystem.systemState, config['domain'], waveSystem.waveState.v, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'vanimo', markerSize = markerSize, gridVisualization = False, title = 'v [Initial]')

uPlotRk4 = visualizeParticles(fig, axis[0,1], waveSystem.systemState, config['domain'], waveSystemNext.waveState.u, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'managua', markerSize = markerSize, gridVisualization = False, title = 'u [Prediction]')
vPlotRk4 = visualizeParticles(fig, axis[1,1], waveSystem.systemState, config['domain'], waveSystemNext.waveState.v, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'vanimo', markerSize = markerSize, gridVisualization = False, title = 'v [Prediction]')


uDiff = waveSystem.waveState.u - trajectory[0,:,0]
vDiff = waveSystem.waveState.v - trajectory[0,:,1]

uDiff = trajectory[0,:,0]
vDiff = trajectory[0,:,1]

uPlotGT = visualizeParticles(fig, axis[0,2], waveSystem.systemState, config['domain'], uDiff, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'managua', markerSize = markerSize, gridVisualization = False, title = 'u [Ground Truth]')
vPlotGT = visualizeParticles(fig, axis[1,2], waveSystem.systemState, config['domain'], vDiff, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'vanimo', markerSize = markerSize, gridVisualization = False, title = 'v [Ground Truth]')

diffU = (waveSystemNext.waveState.u - trajectory[0,:,0])#.detach().cpu().numpy()
diffV = (waveSystemNext.waveState.v - trajectory[0,:,1])#.detach().cpu().numpy()

visualizeParticles(fig, axis[0,3], waveSystem.systemState, config['domain'], diffU, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'bwr', markerSize = markerSize, gridVisualization = False, title = 'u [Diff]')
visualizeParticles(fig, axis[1,3], waveSystem.systemState, config['domain'], diffV, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'bwr', markerSize = markerSize, gridVisualization = False, title = 'v [Diff]')


fig.tight_layout()


In [ ]:
fig, axis = plt.subplots(1, 1)
axis.plot(losses, label='Total Loss')
axis.plot([u for u in uLosses], label='u Loss')
axis.plot([v for v in vLosses], label='v Loss')
axis.set_yscale('log')
axis.set_title('Loss')
axis.set_xlabel('Iteration')
axis.set_ylabel('Loss')
axis.grid(True)
axis.legend()

fig.tight_layout()

In [ ]:
frameStart = datasetLoader.dataset[24]

def unsq(t: Union[torch.Tensor, float, int], device) -> torch.Tensor:
    # print(f'Unsq input: {t}, type: {type(t)}')
    if isinstance(t, torch.Tensor):
        return t.unsqueeze(0)
    elif isinstance(t, str):
        return t
    else:
        return torch.tensor([t], device=device)
    

def batchifyFrame(frame):
    historyState, targetState, currentState, positions, densities, supports, volumes, dt, scheme, fileNames, simIndex, startingPoint = frame
    device = historyState.device
    
    batch = (
        unsq(historyState, device),
        unsq(targetState, device),
        unsq(currentState, device),
        unsq(positions, device),
        unsq(densities, device),
        unsq(supports, device),
        unsq(volumes, device),
        unsq(dt, device),
        unsq(scheme, device),
        unsq(fileNames, device),
        unsq(simIndex, device),        unsq(startingPoint, device),
    )
    
    return batch

batchedFrame = batchifyFrame(frameStart)

In [ ]:
waveSystem, config, integrator, dt, history, trajectory, current = batchToSimulation(batchedFrame)

In [ ]:
eulerIntegrator = getIntegrator(IntegrationSchemeType.explicitEuler)
RK4Integrator = getIntegrator(IntegrationSchemeType.rungeKutta4)

In [ ]:
n = waveSystem.waveState.u.shape[0]
nx = int(n**0.5)
print(f'Assuming square grid with nx={nx}, ny={nx}')
domainArea = (config['domain'].max[0] - config['domain'].min[0]) * (config['domain'].max[1] - config['domain'].min[1])
print(f'Particle area: {domainArea/n}, particle spacing: {(domainArea/n)**0.5}')
dx = (domainArea/n)**0.5


In [ ]:

waveSystemEuler = copyWaveSystem(waveSystem)
waveSystemRK4 = copyWaveSystem(waveSystem)
waveSystemNeural = copyWaveSystem(waveSystem)

for i in range(1):
    waveSystemEuler, updates =  eulerIntegrator.function(
        waveSystemEuler,
        dt = dt,
        f = waveSystemFunction,
        verbose = False,
        config = config,
    )
    waveSystemRK4, updates = RK4Integrator.function(
        waveSystemRK4,
        dt = dt,
        f = waveSystemFunction,
        verbose = False,
        config = config,    
    )

    waveSystemNext = copyWaveSystem(waveSystemNeural)

    features = torch.cat([
        waveSystemNext.waveState.u.view(-1, 1),
        waveSystemNext.waveState.v.view(-1, 1),
    ], dim=-1)

    with torch.no_grad():
        fi_nn = GNN(waveSystemNext.systemState, waveSystemNext.neighborhood, features)

    dudt = fi_nn[:, 0]
    dvdt = fi_nn[:, 1]

    waveSystemNeural.waveState.u = waveSystemNext.waveState.u + dudt
    waveSystemNeural.waveState.v = waveSystemNext.waveState.v + dvdt
    
    

In [ ]:
fig, axis = plt.subplots(2, 5, figsize=(18,6), sharex=True, sharey=True)

markerSize = 1.5

uMaxRk4 = torch.max(torch.abs(waveSystemRK4.waveState.u)).cpu().detach().item()
vMaxRk4 = torch.max(torch.abs(waveSystemRK4.waveState.v)).cpu().detach().item()

uPlotRk4 = visualizeParticles(fig, axis[0,0], waveSystem.systemState, config['domain'], waveSystemRK4.waveState.u, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'managua', markerSize = markerSize, gridVisualization = False, title = 'u [Rk4]', vmin = -uMaxRk4, vmax = uMaxRk4)
vPlotRk4 = visualizeParticles(fig, axis[1,0], waveSystem.systemState, config['domain'], waveSystemRK4.waveState.v, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'vanimo', markerSize = markerSize, gridVisualization = False, title = 'v [Rk4]', vmin = -vMaxRk4, vmax = vMaxRk4)


umaxEuler = torch.max(torch.abs(waveSystemNeural.waveState.u)).cpu().detach().item()
vmaxEuler = torch.max(torch.abs(waveSystemNeural.waveState.v)).cpu().detach().item()

uPlotEuler = visualizeParticles(fig, axis[0,1], waveSystem.systemState, config['domain'], waveSystemNeural.waveState.u, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'managua', markerSize = markerSize, gridVisualization = False, title = 'u [Neural]', vmin = -umaxEuler, vmax = umaxEuler)
vPlotEuler = visualizeParticles(fig, axis[1,1], waveSystem.systemState, config['domain'], waveSystemNeural.waveState.v, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'vanimo', markerSize = markerSize, gridVisualization = False, title = 'v [Neural]', vmin = -vmaxEuler, vmax = vmaxEuler)

uDiff = (waveSystemRK4.waveState.u - waveSystemNeural.waveState.u)
vDiff = (waveSystemRK4.waveState.v - waveSystemNeural.waveState.v)

uDiff = torch.stack([u.dudt for u in updates], dim=0).mean(dim=0) * dt
vDiff = torch.stack([u.dvdt for u in updates], dim=0).mean(dim=0) * dt

uDiffVmax = torch.max(torch.abs(uDiff)).cpu().detach().item()
vDiffVmax = torch.max(torch.abs(vDiff)).cpu().detach().item()

uPlotDiff = visualizeParticles(fig, axis[0,2], waveSystem.systemState, config['domain'], uDiff, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'RdBu', markerSize = markerSize, gridVisualization = False, title = 'du [RK4]', vmin = -uDiffVmax, vmax = uDiffVmax)
vPlotDiff = visualizeParticles(fig, axis[1,2], waveSystem.systemState, config['domain'], vDiff, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'RdBu', markerSize = markerSize, gridVisualization = False, title = 'dv [RK4]', vmin = -vDiffVmax, vmax = vDiffVmax)

# uDiff = (waveSystemRK4.waveState.u - waveSystem.waveState.u)
# vDiff = (waveSystemRK4.waveState.v - waveSystem.waveState.v)
uDiffNN = fi_nn[:, 0]
vDiffNN = fi_nn[:, 1]

uDiffVmax = torch.max(torch.abs(uDiffNN)).cpu().detach().item()
vDiffVmax = torch.max(torch.abs(vDiffNN)).cpu().detach().item()

uPlotDiffInitial = visualizeParticles(fig, axis[0,3], waveSystem.systemState, config['domain'], uDiffNN, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'RdBu', markerSize = markerSize, gridVisualization = False, title = 'du [Prediction]', vmin = -uDiffVmax, vmax = uDiffVmax)
vPlotDiffInitial = visualizeParticles(fig, axis[1,3], waveSystem.systemState, config['domain'], vDiffNN, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'RdBu', markerSize = markerSize, gridVisualization = False, title = 'dv [Prediction]', vmin = -vDiffVmax, vmax = vDiffVmax)

errU = uDiff - uDiffNN
errV = vDiff - vDiffNN

errUmax = torch.max(torch.abs(errU)).cpu().detach().item()
errVmax = torch.max(torch.abs(errV)).cpu().detach().item()

uPlotError = visualizeParticles(fig, axis[0,4], waveSystem.systemState, config['domain'], errU, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'RdBu', markerSize = markerSize, gridVisualization = False, title = 'du [Error]', vmin = -errUmax, vmax = errUmax)
vPlotError = visualizeParticles(fig, axis[1,4], waveSystem.systemState, config['domain'], errV, config['kernel'], which = 'both', visualizeBoth = True, cbar = True, cmap = 'RdBu', markerSize = markerSize, gridVisualization = False, title = 'dv [Error]', vmin = -errVmax, vmax = errVmax)


fig.tight_layout()